# DeepLabV3+ v9 — Birleşik En İyi Konfigürasyon

**v9 = v7-c4fix taban + 384×384 + multi-scale TTA**

| Özellik | v7-fixed | v7-c4fix | v8-exp | **v9** |
|---|---|---|---|---|
| Çözünürlük | 320×320 | 320×320 | 384×384 | **384×384** |
| 2.5D komşu | ±1 (3 kanal) | **±2 (5 kanal)** | ±1 | **±2 (5 kanal)** |
| Loss | 3'lü | **4'lü + Lovász** | 3'lü | **4'lü + Lovász** |
| xline aug | aynı | **agresif elastic** | aynı | **agresif elastic** |
| TTA | HFlip+pol | HFlip+pol | **+scale 0.75/1.25** | **+scale 0.75/1.25** |
| Batch / Accum | 6 / 4 | 6 / 4 | 4 / 6 | **4 / 6** |

**Hedef:** Combined mIoU > 0.79 ve Class 4 Test2 IoU > 0.20 (v7-c4fix bazı: 0.7779 / 0.1820).


## 0. Kurulum ve Cihaz

In [18]:
import os, subprocess, sys

# HF Hub progress bar (ipywidget) ve symlink uyarilarini en bastan kapat
# (cell-14'te EfficientNet-B4 indirilirken VSCode'un ipywidget renderer'i takiliyor)
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"]    = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TQDM_DISABLE"]                    = "1"

pkgs = [
    "matplotlib",
    "scikit-learn",
    "albumentations",
    "segmentation-models-pytorch",
    "ipywidgets",
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + pkgs)
print("Kurulum tamamlandi.")

Kurulum tamamlandi.


In [19]:
import os, sys, random, json, time, zipfile
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset, WeightedRandomSampler
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from sklearn.metrics import confusion_matrix
import segmentation_models_pytorch as smp
import albumentations as A

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Apple Silicon MPS backend aktif")
else:
    device = torch.device("cpu")
    print("UYARI: GPU bulunamadi, CPU kullaniliyor!")
print(f"Cihaz: {device} | PyTorch: {torch.__version__} | SMP: {smp.__version__}")

NOTEBOOK_DIR = Path(".").resolve()
PROJECT_DIR  = NOTEBOOK_DIR
DATA_DIR     = PROJECT_DIR / "data"
CHECKPOINTS  = PROJECT_DIR / "checkpoints_v7"
FIGURES_DIR  = PROJECT_DIR / "results" / "figures"
METRICS_DIR  = PROJECT_DIR / "results" / "metrics"

for d in [DATA_DIR, CHECKPOINTS, FIGURES_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Proje : {PROJECT_DIR}")
print(f"Veri  : {DATA_DIR}")
print(f"  train_seismic.npy mevcut: {(DATA_DIR / 'train' / 'train_seismic.npy').exists()}")

GPU: NVIDIA GeForce RTX 3060 Ti
VRAM: 8.6 GB
Cihaz: cuda | PyTorch: 2.6.0+cu124 | SMP: 0.5.0
Proje : C:\Users\Teknogenetik\Desktop\sismik-proje-main
Veri  : C:\Users\Teknogenetik\Desktop\sismik-proje-main\data
  train_seismic.npy mevcut: True


## 1. Veri Yukleme

In [20]:
TRAIN_SEISMIC = DATA_DIR / "train" / "train_seismic.npy"

if not TRAIN_SEISMIC.exists():
    zip_path = DATA_DIR / "data.zip"
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("Veri indiriliyor (~1 GB)...")

    def _progress(block, block_size, total):
        downloaded = block * block_size
        if total > 0:
            pct = min(100, downloaded * 100 / total)
            print(f"\r  {pct:.1f}%  ({downloaded/1e6:.0f} MB)", end="", flush=True)

    urlretrieve("https://zenodo.org/record/3755060/files/data.zip",
                zip_path, reporthook=_progress)
    print("\nCikartiliyor...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(DATA_DIR)
    zip_path.unlink()
    print("Hazir.")
else:
    print("Veri zaten mevcut.")

print("NPY yukleniyor...")
train_seismic = np.load(DATA_DIR / "train"     / "train_seismic.npy")
train_labels  = np.load(DATA_DIR / "train"     / "train_labels.npy")
test1_seismic = np.load(DATA_DIR / "test_once" / "test1_seismic.npy")
test1_labels  = np.load(DATA_DIR / "test_once" / "test1_labels.npy")
test2_seismic = np.load(DATA_DIR / "test_once" / "test2_seismic.npy")
test2_labels  = np.load(DATA_DIR / "test_once" / "test2_labels.npy")

print(f"Train volume : {train_seismic.shape}  dtype={train_seismic.dtype}")
print(f"  -> Inline sayisi  : {train_seismic.shape[0]}")
print(f"  -> Crossline sayisi: {train_seismic.shape[1]}")
print(f"  -> Derinlik       : {train_seismic.shape[2]}")
print(f"Test1 (inline)    : {test1_seismic.shape}")
print(f"Test2 (crossline) : {test2_seismic.shape}")

Veri zaten mevcut.
NPY yukleniyor...
Train volume : (401, 701, 255)  dtype=float64
  -> Inline sayisi  : 401
  -> Crossline sayisi: 701
  -> Derinlik       : 255
Test1 (inline)    : (200, 701, 255)
Test2 (crossline) : (601, 200, 255)


## 2. EDA

In [21]:
CLASS_NAMES   = ["Upper NS", "Lower NS", "Rijnland", "Scruff", "Zechstein", "Under Zech"]
NUM_CLASSES   = 6
FACIES_COLORS = ["#3288bd", "#66c2a5", "#abdda4", "#e6f598", "#fdae61", "#f46d43"]
cmap_facies   = mcolors.ListedColormap(FACIES_COLORS)

counts = np.bincount(train_labels.flatten(), minlength=NUM_CLASSES)
total  = counts.sum()

print("Sinif dagilimi (Train volume):")
for i, (name, cnt) in enumerate(zip(CLASS_NAMES, counts)):
    print(f"  S{i} {name:22s}: {cnt:>12,}  ({cnt/total*100:.2f}%)")

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

bars = axes[0,0].bar(range(NUM_CLASSES), counts / 1e6, color=FACIES_COLORS, edgecolor="k", lw=0.5)
axes[0,0].set_xticks(range(NUM_CLASSES))
axes[0,0].set_xticklabels([f"S{j}\n{CLASS_NAMES[j]}" for j in range(NUM_CLASSES)], fontsize=8)
axes[0,0].set_ylabel("Piksel (Milyon)")
axes[0,0].set_title("Sinif Dagilimi")

idx_inl = train_seismic.shape[0] // 2
axes[0,1].imshow(train_seismic[idx_inl].T, cmap="seismic", aspect="auto", vmin=-1, vmax=1)
axes[0,1].set_title(f"Inline #{idx_inl}")

idx_xln = train_seismic.shape[1] // 2
axes[1,0].imshow(train_seismic[:, idx_xln, :].T, cmap="seismic", aspect="auto", vmin=-1, vmax=1)
axes[1,0].set_title(f"Crossline #{idx_xln}")

axes[1,1].imshow(train_labels[idx_inl].T, cmap=cmap_facies, vmin=-0.5, vmax=5.5, aspect="auto")
axes[1,1].set_title(f"Fasiyes — Inline #{idx_inl}")

plt.suptitle("F3 Block — Inline ve Crossline Kesitler", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "eda_multiview.png", dpi=150, bbox_inches="tight")
plt.show()

Sinif dagilimi (Train volume):
  S0 Upper NS              :   20,137,839  (28.09%)
  S1 Lower NS              :    8,519,666  (11.89%)
  S2 Rijnland              :   34,831,122  (48.59%)
  S3 Scruff                :    4,760,778  (6.64%)
  S4 Zechstein             :    2,350,150  (3.28%)
  S5 Under Zech            :    1,081,200  (1.51%)


C:\Users\Teknogenetik\AppData\Local\Temp\ipykernel_19196\4191152698.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Methodology-Fix Edilmis Train/Val Split + Sinif Agirliklari (Yol A)

Eski split'in 3 bug'i vardi:
1. Val son %20 contiguous blok → lokasyon bias
2. Train xline'lar val inline pikselleri iceriyordu → 3D leakage
3. Val sinirinda 2.5D komsu sizintisi (val[320]'nin kanali = [319,320,321], 319 train'de)

Yol A duzeltmeleri:
- **Val ortaya kaydirildi** (inline 160–240) — block surrounded by train, lokasyon bias zayifladi
- **±2 inline buffer** — train val sinirina 2 inline yaklasamaz (2.5D komsu fix)
- **Xline image'lar val pikselleri haric cropped** (cell-11'de implemented) — 3D leakage giderildi

In [22]:
n_inlines = train_seismic.shape[0]
n_crosslines = train_seismic.shape[1]

# METHODOLOGY FIX (Yol A) — orta blok val + buffer + xline cropping mask
BUFFER = 2
val_start, val_end = 160, 240   # ortadaki 80 inline
val_inline_idx = np.arange(val_start, val_end)
train_inline_idx = np.concatenate([
    np.arange(0, val_start - BUFFER),
    np.arange(val_end + BUFFER, n_inlines)
])
train_xline_idx = np.arange(0, n_crosslines)

# Xline image cropping icin boolean mask (val + buffer haric)
train_inline_mask = np.zeros(n_inlines, dtype=bool)
train_inline_mask[train_inline_idx] = True

print(f"Train inline : {len(train_inline_idx)} slice  (iki blok, ±{BUFFER} buffer ile)")
print(f"  Blok 1: [0, {val_start - BUFFER})")
print(f"  Blok 2: [{val_end + BUFFER}, {n_inlines})")
print(f"Val inline   : [{val_start}, {val_end}) -> {len(val_inline_idx)} slice (ortada)")
print(f"Train xline  : {len(train_xline_idx)} slice (val pikselleri image'dan cropped)")
print(f"Toplam train : {len(train_inline_idx) + len(train_xline_idx)} slice")

# Class weights yalnizca train inline'lardan hesaplanir (val + buffer haric)
train_counts = np.bincount(
    train_labels[train_inline_idx].flatten(), minlength=NUM_CLASSES
).astype(float)
class_freq = train_counts / train_counts.sum()
focal_alpha = 1.0 / (class_freq + 1e-8)
focal_alpha = focal_alpha / focal_alpha.sum()
focal_alpha_t = torch.FloatTensor(focal_alpha).to(device)

print("\nSinif frekanslari ve Focal alpha (train inline blok'larindan):")
for i, (name, f, a) in enumerate(zip(CLASS_NAMES, class_freq, focal_alpha)):
    print(f"  S{i} {name:22s}: freq={f:.4f}  alpha={a:.4f}")

Train inline : 317 slice  (iki blok, ±2 buffer ile)
  Blok 1: [0, 158)
  Blok 2: [242, 401)
Val inline   : [160, 240) -> 80 slice (ortada)
Train xline  : 701 slice (val pikselleri image'dan cropped)
Toplam train : 1018 slice

Sinif frekanslari ve Focal alpha (train inline blok'larindan):
  S0 Upper NS              : freq=0.2783  alpha=0.0329
  S1 Lower NS              : freq=0.1191  alpha=0.0768
  S2 Rijnland              : freq=0.4827  alpha=0.0190
  S3 Scruff                : freq=0.0637  alpha=0.1436
  S4 Zechstein             : freq=0.0371  alpha=0.2465
  S5 Under Zech            : freq=0.0190  alpha=0.4813


## 4. 2.5D Dataset + Multi-View DataLoader + Rare-Class Sampling

In [23]:
IMG_SIZE     = (384, 384)            # v9: 320 -> 384
BATCH_SIZE   = 4                       # v9: 6 -> 4 (384x384 + 5ch icin VRAM)
ACCUM_STEPS  = 6                       # v9: 4 -> 6 (efektif batch = 24 sabit)
MIXUP_ALPHA  = 0.2
N_NEIGHBORS  = 2        # v7-c4fix: ±2 komsu (eski: 1)
N_CHANNELS   = 2 * N_NEIGHBORS + 1   # = 5

# Genel augmentation (inline + val icin)
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=10,
                       border_mode=0, p=0.5),
    A.ElasticTransform(alpha=80, sigma=10, p=0.3),
    A.GridDistortion(num_steps=5, distort_limit=0.2, p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
    A.GaussNoise(std_range=(0.001, 0.015), p=0.3),
    A.CoarseDropout(max_holes=6, max_height=20, max_width=20, fill=0, p=0.2),
])

# Crossline-aware augmentation — Class 4 morfolojisi xline'da kivrimli/diapir
# Daha agresif elastic + grid distortion ile salt'in xline goruntusunu zorla
xline_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.12, scale_limit=0.20, rotate_limit=12,
                       border_mode=0, p=0.6),
    A.ElasticTransform(alpha=140, sigma=12, p=0.55),    # boost
    A.GridDistortion(num_steps=6, distort_limit=0.30, p=0.50),  # boost
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
    A.GaussNoise(std_range=(0.001, 0.015), p=0.3),
    A.CoarseDropout(max_holes=6, max_height=20, max_width=20, fill=0, p=0.2),
])


class F3Dataset25D(Dataset):
    """
    2.5D Dataset: ±n_neighbors komsu slice -> (2*n_neighbors+1) kanalli input.
    v7-c4fix default: n_neighbors=2 -> 5 kanal.

    METHODOLOGY FIX (Yol A):
    train_inline_mask: val pikselleri xline image'dan cikarilmasi icin maske.
    """
    def __init__(self, volume, labels, indices, axis=0, img_size=IMG_SIZE,
                 transform=None, augment_polarity=False, train_inline_mask=None,
                 n_neighbors=N_NEIGHBORS):
        self.volume   = volume
        self.labels   = labels
        self.indices  = indices
        self.axis     = axis
        self.n_slices = volume.shape[axis]
        self.img_size = img_size
        self.transform = transform
        self.augment_polarity = augment_polarity
        self.train_inline_mask = train_inline_mask
        self.n_neighbors = n_neighbors

    def _get_slice(self, vol, idx):
        if self.axis == 0:
            return vol[idx]
        else:
            s = vol[:, idx]
            if self.train_inline_mask is not None:
                s = s[self.train_inline_mask]
            return s

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        # ±n komsu slice topla, sinirda klampla
        slices = []
        for k in range(-self.n_neighbors, self.n_neighbors + 1):
            j = max(0, min(self.n_slices - 1, i + k))
            slices.append(self._get_slice(self.volume, j).astype(np.float32))
        img  = np.stack(slices, axis=-1)   # (H, W, C)
        mask = self._get_slice(self.labels, i).astype(np.int64)

        if self.augment_polarity and random.random() > 0.5:
            img = -img

        if self.transform is not None:
            aug  = self.transform(image=img, mask=mask.astype(np.uint8))
            img  = aug["image"]
            mask = aug["mask"].astype(np.int64)

        img_t  = torch.from_numpy(img.copy()).permute(2, 0, 1).float()
        mask_t = torch.from_numpy(mask.copy())

        img_t  = F.interpolate(img_t.unsqueeze(0), size=self.img_size,
                               mode="bilinear", align_corners=False).squeeze(0)
        mask_t = F.interpolate(mask_t.float().unsqueeze(0).unsqueeze(0),
                               size=self.img_size, mode="nearest").squeeze(0).squeeze(0).long()
        return img_t, mask_t


# Volume normalize
train_mean = train_seismic.mean()
train_std  = train_seismic.std() + 1e-8
train_seis_norm = ((train_seismic - train_mean) / train_std).astype(np.float32)
test1_seis_norm = ((test1_seismic - train_mean) / train_std).astype(np.float32)
test2_seis_norm = ((test2_seismic - train_mean) / train_std).astype(np.float32)

# Dataset'ler
train_inline_ds = F3Dataset25D(
    train_seis_norm, train_labels, train_inline_idx,
    axis=0, transform=train_transform, augment_polarity=True)

train_xline_ds = F3Dataset25D(
    train_seis_norm, train_labels, train_xline_idx,
    axis=1, transform=xline_transform, augment_polarity=True,   # xline-aware aug
    train_inline_mask=train_inline_mask)

train_ds = ConcatDataset([train_inline_ds, train_xline_ds])

val_ds = F3Dataset25D(train_seis_norm, train_labels, val_inline_idx, axis=0)

test1_ds = F3Dataset25D(test1_seis_norm, test1_labels, np.arange(test1_seismic.shape[0]), axis=0)
test2_ds = F3Dataset25D(test2_seis_norm, test2_labels, np.arange(test2_seismic.shape[1]), axis=1)

print(f"Train inline : {len(train_inline_ds)} slice")
print(f"Train xline  : {len(train_xline_ds)} slice (xline-aware aug)")
print(f"Train toplam : {len(train_ds)} slice")
print(f"Val          : {len(val_ds)} slice (orta blok)")
print(f"Test1        : {len(test1_ds)} slice")
print(f"Test2        : {len(test2_ds)} slice (ayri volume)")
print(f"IMG_SIZE     : {IMG_SIZE}  |  Kanal: {N_CHANNELS} (n_neighbors={N_NEIGHBORS})")
print(f"Mixup alpha  : {MIXUP_ALPHA}")


c:\Users\Teknogenetik\Desktop\sismik-proje-main\.venv\Lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
C:\Users\Teknogenetik\AppData\Local\Temp\ipykernel_19196\4133812320.py:17: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=6, max_height=20, max_width=20, fill=0, p=0.2),
C:\Users\Teknogenetik\AppData\Local\Temp\ipykernel_19196\4133812320.py:30: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=6, max_height=20, max_width=20, fill=0, p=0.2),


Train inline : 317 slice
Train xline  : 701 slice (xline-aware aug)
Train toplam : 1018 slice
Val          : 80 slice (orta blok)
Test1        : 200 slice
Test2        : 200 slice (ayri volume)
IMG_SIZE     : (384, 384)  |  Kanal: 5 (n_neighbors=2)
Mixup alpha  : 0.2


In [24]:
RARE_CLASSES = [4, 5]
RARE_BOOST   = 10.0

def compute_slice_weights(labels, indices, axis, rare_classes=RARE_CLASSES,
                          boost=RARE_BOOST, train_inline_mask=None):
    """
    Slice basina rare-class boost agirligi.
    train_inline_mask gecirildiginde xline (axis=1) image'lar val piksellerini
    icermez — sampling agirligi gercek egitim image'i ile tutarlidir.
    """
    weights = []
    for i in indices:
        if axis == 0:
            sl = labels[i]
        else:
            sl = labels[:, i]
            if train_inline_mask is not None:
                sl = sl[train_inline_mask]
        total_px = sl.size
        rare_px  = sum(int((sl == c).sum()) for c in rare_classes)
        w = 1.0 + (rare_px / total_px) * boost
        weights.append(w)
    return weights

w_inline = compute_slice_weights(train_labels, train_inline_idx, axis=0)
w_xline  = compute_slice_weights(train_labels, train_xline_idx, axis=1,
                                 train_inline_mask=train_inline_mask)
all_weights = w_inline + w_xline

sampler = WeightedRandomSampler(
    weights=all_weights,
    num_samples=len(train_ds),
    replacement=True
)

w_arr = np.array(all_weights)
print(f"Sampling agirliklari: min={w_arr.min():.2f}  max={w_arr.max():.2f}  "
      f"mean={w_arr.mean():.2f}  median={np.median(w_arr):.2f}")
print(f"Agirlik > 1.5 olan slice'lar: {(w_arr > 1.5).sum()} / {len(w_arr)}  "
      f"({(w_arr > 1.5).mean()*100:.1f}%)")

NUM_WORKERS = 0 if sys.platform == "win32" else 2
pin = device.type == "cuda"

train_loader = DataLoader(train_ds,  batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=pin)
val_loader   = DataLoader(val_ds,    batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=pin)
test1_loader = DataLoader(test1_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test2_loader = DataLoader(test2_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

imgs, masks = next(iter(train_loader))
print(f"\nBatch — goruntu: {imgs.shape}  maske: {masks.shape}")
print(f"  -> 3 kanal = [slice-1, slice, slice+1] (2.5D)")
print(f"Efektif batch size: {BATCH_SIZE} x {ACCUM_STEPS} = {BATCH_SIZE * ACCUM_STEPS}")

Sampling agirliklari: min=1.00  max=2.42  mean=1.56  median=1.51
Agirlik > 1.5 olan slice'lar: 522 / 1018  (51.3%)

Batch — goruntu: torch.Size([4, 5, 384, 384])  maske: torch.Size([4, 384, 384])
  -> 3 kanal = [slice-1, slice, slice+1] (2.5D)
Efektif batch size: 4 x 6 = 24


## 5. Model — DeepLabV3+ (EfficientNet-B4, in_channels=3)
3 kanalli 2.5D input → ImageNet pretrained agirliklar dogrudan uyumlu.  
ASPP rates (12, 24, 36) → genis receptive field, jeolojik katman gecislerini yakalar.

In [25]:
model = smp.DeepLabV3Plus(
    encoder_name="efficientnet-b4",
    encoder_weights="imagenet",
    in_channels=N_CHANNELS,                 # 5 (v7-c4fix)
    classes=NUM_CLASSES,
    activation=None,
    encoder_output_stride=16,
    decoder_atrous_rates=(12, 24, 36),
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Toplam parametre: {n_params:,}")
print(f"Encoder: EfficientNet-B4 | in_channels={N_CHANNELS} | ASPP rates=(12,24,36)")
print(f"Not: smp first_conv'u kanal ortalamasi ile {N_CHANNELS} kanala genisletti (ImageNet weights korundu)")

with torch.no_grad():
    dummy = torch.randn(2, N_CHANNELS, 384, 384).to(device)
    out = model(dummy)
    print(f"Cikti boyutu: {out.shape}  (beklenen: [2, {NUM_CLASSES}, 384, 384])")
del dummy, out
if device.type == 'cuda':
    torch.cuda.empty_cache()
print("Model hazir.")


Toplam parametre: 18,618,366
Encoder: EfficientNet-B4 | in_channels=5 | ASPP rates=(12,24,36)
Not: smp first_conv'u kanal ortalamasi ile 5 kanala genisletti (ImageNet weights korundu)
Cikti boyutu: torch.Size([2, 6, 384, 384])  (beklenen: [2, 6, 384, 384])
Model hazir.


## 6. Loss (0.4 LS-CE + 0.3 Dice + 0.3 Focal) + Mixup + AdamW + CosineWR

In [ ]:
# ─── Mixup fonksiyonu (v5'ten) ───
def mixup_data(x, y, alpha=0.2):
    """Mixup augmentation: rastgele cift interpolasyonu."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


class FocalLoss(nn.Module):
    def __init__(self, alpha, gamma=2.0):
        super().__init__()
        self.register_buffer('alpha', alpha)
        self.gamma = gamma

    def forward(self, pred, target):
        ce = F.cross_entropy(pred, target, reduction='none')
        pt = torch.exp(-ce)
        alpha_t = self.alpha[target]
        loss = alpha_t * ((1 - pt) ** self.gamma) * ce
        return loss.mean()


class DiceLoss(nn.Module):
    def __init__(self, n_classes=NUM_CLASSES, smooth=1.0):
        super().__init__()
        self.n_classes = n_classes; self.smooth = smooth

    def forward(self, pred, target):
        pred_soft = torch.softmax(pred, dim=1)
        target_oh = F.one_hot(target, self.n_classes).permute(0, 3, 1, 2).float()
        dice = 0.0
        for c in range(self.n_classes):
            p = pred_soft[:, c]; t = target_oh[:, c]
            inter = (p * t).sum()
            dice += (2 * inter + self.smooth) / (p.sum() + t.sum() + self.smooth)
        return 1 - dice / self.n_classes


# ─── Lovász-Softmax (Berman et al. CVPR 2018) — IoU surrogate ───
def _lovasz_grad(gt_sorted):
    """Lovasz extension w.r.t. sorted errors."""
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jaccard = 1. - intersection / union
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard


def _lovasz_softmax_flat(probas, labels, classes='present'):
    """Multi-class Lovasz-Softmax, flattened."""
    if probas.numel() == 0:
        return probas * 0.
    C = probas.size(1)
    losses = []
    class_to_sum = list(range(C)) if classes in ['all', 'present'] else classes
    for c in class_to_sum:
        fg = (labels == c).float()
        if classes == 'present' and fg.sum() == 0:
            continue
        errors = (fg - probas[:, c]).abs()
        errors_sorted, perm = torch.sort(errors, 0, descending=True)
        fg_sorted = fg[perm]
        losses.append(torch.dot(errors_sorted, _lovasz_grad(fg_sorted)))
    if not losses:
        return probas.sum() * 0.
    return torch.stack(losses).mean()


class LovaszSoftmax(nn.Module):
    """Lovasz-Softmax loss — IoU'yu surrogate olarak optimize eder."""
    def __init__(self, classes='present'):
        super().__init__()
        self.classes = classes

    def forward(self, logits, labels):
        # FP32 zorla — Lovasz sort+cumsum AMP ile 384x384'te NaN uretiyor
        with torch.amp.autocast(device_type='cuda', enabled=False):
            logits = logits.float()
            probas = torch.softmax(logits, dim=1)
            B, C, H, W = probas.shape
            probas_flat = probas.permute(0, 2, 3, 1).reshape(-1, C)
            labels_flat = labels.reshape(-1)
            return _lovasz_softmax_flat(probas_flat, labels_flat, self.classes)


class QuadrupleLoss(nn.Module):
    """v7-c4fix: 0.30 LS-CE + 0.25 Dice + 0.25 Focal + 0.20 Lovasz."""
    def __init__(self, alpha, gamma=2.0, label_smoothing=0.1):
        super().__init__()
        self.focal   = FocalLoss(alpha, gamma)
        self.dice    = DiceLoss()
        self.lovasz  = LovaszSoftmax(classes='present')
        self.label_smoothing = label_smoothing

    def forward(self, pred, target):
        # v9 hotfix: AMP + 384x384 fp16 NaN — tum forward FP32de
        with torch.amp.autocast(device_type='cuda', enabled=False):
            pred = pred.float()
            ls_ce = F.cross_entropy(pred, target, label_smoothing=self.label_smoothing)
            return (0.30 * ls_ce
                    + 0.25 * self.dice(pred, target)
                    + 0.25 * self.focal(pred, target)
                    + 0.20 * self.lovasz(pred, target))


NUM_EPOCHS = 100
criterion  = QuadrupleLoss(focal_alpha_t, gamma=2.0, label_smoothing=0.1).to(device)
optimizer  = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=25,
    T_mult=1,
    eta_min=1e-6,
)

print("Loss  : 0.30*LS-CE(eps=0.1) + 0.25*Dice + 0.25*Focal(g=2) + 0.20*Lovasz-Softmax")
print("Mixup : alpha=0.2, p=0.5 per batch")
print("Optim : AdamW  lr=1e-4  wd=1e-3")
print("Sched : CosineAnnealingWarmRestarts  T_0=25  eta_min=1e-6")
print(f"Epoch : {NUM_EPOCHS}")
print(f"Grad Accumulation: {ACCUM_STEPS} steps")


## 7. Metrik Fonksiyonlari

In [27]:
def compute_metrics(preds_all, targets_all, n=NUM_CLASSES):
    res = {}
    res['PA'] = float((preds_all == targets_all).sum() / len(targets_all))
    cm = confusion_matrix(targets_all, preds_all, labels=list(range(n)))

    row_sums  = cm.sum(axis=1).astype(float)
    class_acc = np.where(row_sums > 0, cm.diagonal() / row_sums, 0.0)
    res['MCA'] = float(class_acc.mean())
    res['per_class_acc'] = class_acc.tolist()

    ious = []
    for c in range(n):
        inter = cm[c, c]; union = cm[c,:].sum() + cm[:,c].sum() - inter
        ious.append(float(inter / (union + 1e-8)))
    res['mIoU'] = float(np.mean(ious))
    res['per_class_iou'] = ious

    dices = []
    for c in range(n):
        tp = cm[c,c]; fp = cm[:,c].sum()-tp; fn = cm[c,:].sum()-tp
        dices.append(float(2*tp / (2*tp + fp + fn + 1e-8)))
    res['mean_dice'] = float(np.mean(dices))
    res['per_class_dice'] = dices
    res['confusion_matrix'] = cm.tolist()
    return res


def print_metrics(metrics, title='Model'):
    print(f'\n{"="*60}')
    print(f' {title}')
    print(f'{"="*60}')
    print(f'  PA        : {metrics["PA"]*100:.2f}%')
    print(f'  MCA       : {metrics["MCA"]*100:.2f}%')
    print(f'  mIoU      : {metrics["mIoU"]*100:.2f}%')
    print(f'  Mean Dice : {metrics["mean_dice"]*100:.2f}%')
    print('  Per-class IoU:')
    for i, name in enumerate(CLASS_NAMES):
        iou  = metrics['per_class_iou'][i]
        dice = metrics['per_class_dice'][i]
        print(f"    S{i} {name:20s}: IoU={iou:.4f}  Dice={dice:.4f}")


print("Metrik fonksiyonlari hazir.")

Metrik fonksiyonlari hazir.


## 8. Egitim (100 Epoch + Grad Accum + Early Stop)

> Methodology fix sonrasi yeniden egitim. Checkpoint/best model `_fixed` ekleriyle kaydedilir
> ki eski v7 (bozuk methodology) sonuclari korunsun — sunumda "oncesi vs sonrasi" tablosu icin.

In [28]:
CHECKPOINT_PATH = CHECKPOINTS / "deeplabv3plus_v9_checkpoint.pth"
BEST_MODEL_PATH = CHECKPOINTS / "deeplabv3plus_v9_best.pth"

use_amp = device.type == "cuda"
scaler  = torch.amp.GradScaler("cuda") if use_amp else None

start_epoch   = 0
best_val_miou = 0.0
patience_counter = 0
PATIENCE = 25
history = {'train_loss': [], 'val_loss': [], 'val_miou': [], 'val_dice': [], 'lr': []}

if CHECKPOINT_PATH.exists():
    print("Checkpoint bulundu, devam ediliyor...")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    start_epoch   = ckpt['epoch'] + 1
    best_val_miou = ckpt['best_val_miou']
    history       = ckpt['history']
    patience_counter = ckpt.get('patience_counter', 0)
    if use_amp and 'scaler_state' in ckpt:
        scaler.load_state_dict(ckpt['scaler_state'])
    print(f"Epoch {start_epoch}/{NUM_EPOCHS} — onceki en iyi mIoU: {best_val_miou:.4f}")
else:
    print("Sifirdan basliyor (methodology-fixed split)...")

print(f"\nCihaz: {device}  |  AMP: {use_amp}  |  Kalan epoch: {NUM_EPOCHS - start_epoch}")
print(f"Grad Accum: {ACCUM_STEPS}  |  Early Stop Patience: {PATIENCE}")
print(f"Train: {len(train_ds)} slice (inline+xline, val pikselleri cropped)  |  Val: {len(val_ds)} slice (orta blok)")
print("=" * 80)

for epoch in range(start_epoch, NUM_EPOCHS):
    t0 = time.time()

    # --- TRAIN ---
    model.train()
    optimizer.zero_grad()
    train_loss = 0.0
    for step, (imgs, masks) in enumerate(train_loader):
        imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)

        # Mixup: %50 olasilikla uygula
        use_mixup = random.random() < 0.5
        if use_mixup:
            imgs, masks_a, masks_b, lam = mixup_data(imgs, masks, MIXUP_ALPHA)

        if use_amp:
            with torch.amp.autocast("cuda"):
                preds = model(imgs)
                if use_mixup:
                    loss = (lam * criterion(preds, masks_a) + (1 - lam) * criterion(preds, masks_b)) / ACCUM_STEPS
                else:
                    loss = criterion(preds, masks) / ACCUM_STEPS
            scaler.scale(loss).backward()
        else:
            preds = model(imgs)
            if use_mixup:
                loss = (lam * criterion(preds, masks_a) + (1 - lam) * criterion(preds, masks_b)) / ACCUM_STEPS
            else:
                loss = criterion(preds, masks) / ACCUM_STEPS
            loss.backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            if use_amp:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            optimizer.zero_grad()

        train_loss += loss.item() * ACCUM_STEPS
    train_loss /= len(train_loader)
    scheduler.step()  # CosineAnnealingWarmRestarts — epoch sonunda

    # --- VALIDATION ---
    model.eval()
    val_loss = 0.0
    all_p, all_t = [], []
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)
            if use_amp:
                with torch.amp.autocast("cuda"):
                    preds = model(imgs)
            else:
                preds = model(imgs)
            val_loss += criterion(preds, masks).item()
            all_p.append(preds.argmax(1).cpu().numpy().flatten())
            all_t.append(masks.cpu().numpy().flatten())
    val_loss   /= len(val_loader)
    val_metrics = compute_metrics(np.concatenate(all_p), np.concatenate(all_t))

    lr = optimizer.param_groups[0]['lr']
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_miou'].append(val_metrics['mIoU'])
    history['val_dice'].append(val_metrics['mean_dice'])
    history['lr'].append(lr)

    marker = ""
    if val_metrics['mIoU'] > best_val_miou:
        best_val_miou = val_metrics['mIoU']
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        marker = "  *** BEST ***"
        patience_counter = 0
    else:
        patience_counter += 1

    ckpt_data = {
        'epoch': epoch, 'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'best_val_miou': best_val_miou, 'history': history,
        'patience_counter': patience_counter,
    }
    if use_amp:
        ckpt_data['scaler_state'] = scaler.state_dict()
    torch.save(ckpt_data, CHECKPOINT_PATH)

    miou  = val_metrics['mIoU']
    mdice = val_metrics['mean_dice']
    print(f"Ep {epoch+1:03d}/{NUM_EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | "
          f"mIoU: {miou:.4f} | Dice: {mdice:.4f} | lr: {lr:.2e} | {time.time()-t0:.0f}s{marker}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping — {PATIENCE} epoch boyunca iyilesme yok.")
        break

print(f"\nEgitim tamamlandi! En iyi val mIoU: {best_val_miou:.4f}")

Sifirdan basliyor (methodology-fixed split)...

Cihaz: cuda  |  AMP: True  |  Kalan epoch: 100
Grad Accum: 6  |  Early Stop Patience: 25
Train: 1018 slice (inline+xline, val pikselleri cropped)  |  Val: 80 slice (orta blok)
Ep 001/100 | Train: nan | Val: 0.6027 | mIoU: 0.3180 | Dice: 0.4237 | lr: 9.96e-05 | 53s  *** BEST ***
Ep 002/100 | Train: 0.5393 | Val: -56.7108 | mIoU: 0.4768 | Dice: 0.5681 | lr: 9.84e-05 | 52s  *** BEST ***
Ep 003/100 | Train: 0.4027 | Val: -44.9018 | mIoU: 0.4763 | Dice: 0.5599 | lr: 9.65e-05 | 51s
Ep 004/100 | Train: 0.3624 | Val: 0.3288 | mIoU: 0.5078 | Dice: 0.5954 | lr: 9.39e-05 | 51s  *** BEST ***


KeyboardInterrupt: 

## 9. Test Degerlendirmesi + TTA (HFlip + Polarity)

In [ ]:
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device, weights_only=True))
model.eval()
print(f"En iyi model yuklendi: {BEST_MODEL_PATH}")


# v9: Multi-scale TTA — HFlip + polarity + scale 0.75/1.0/1.25
TTA_SCALES = [0.75, 1.25]   # ana scale 1.0 zaten var

def run_eval_tta(loader, tta=True):
    all_p, all_t = [], []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs_dev = imgs.to(device, non_blocking=True)
            H, W = imgs_dev.shape[2], imgs_dev.shape[3]

            # Scale 1.0 — orijinal
            if use_amp:
                with torch.amp.autocast("cuda"):
                    logits = model(imgs_dev)
            else:
                logits = model(imgs_dev)
            probs = torch.softmax(logits, dim=1)
            n_aug = 1

            if tta:
                # HFlip @ scale 1.0
                imgs_hf = torch.flip(imgs_dev, dims=[3])
                if use_amp:
                    with torch.amp.autocast("cuda"):
                        logits_hf = model(imgs_hf)
                else:
                    logits_hf = model(imgs_hf)
                probs += torch.softmax(torch.flip(logits_hf, dims=[3]), dim=1)
                n_aug += 1

                # Polarity inversion @ scale 1.0
                imgs_neg = -imgs_dev
                if use_amp:
                    with torch.amp.autocast("cuda"):
                        logits_neg = model(imgs_neg)
                else:
                    logits_neg = model(imgs_neg)
                probs += torch.softmax(logits_neg, dim=1)
                n_aug += 1

                # Multi-scale: 0.75x ve 1.25x — encoder stride=32 icin /32 yuvarla
                for scale in TTA_SCALES:
                    h_s = max(32, (int(H * scale) // 32) * 32)
                    w_s = max(32, (int(W * scale) // 32) * 32)
                    imgs_s = F.interpolate(imgs_dev, size=(h_s, w_s),
                                           mode="bilinear", align_corners=False)
                    if use_amp:
                        with torch.amp.autocast("cuda"):
                            logits_s = model(imgs_s)
                    else:
                        logits_s = model(imgs_s)
                    logits_s_up = F.interpolate(logits_s, size=(H, W),
                                                mode="bilinear", align_corners=False)
                    probs += torch.softmax(logits_s_up, dim=1)
                    n_aug += 1

                probs /= n_aug   # toplamda 5 augmentation: orig + hflip + polarity + scale0.75 + scale1.25

            all_p.append(probs.argmax(1).cpu().numpy().flatten())
            all_t.append(masks.numpy().flatten())
    return np.concatenate(all_p), np.concatenate(all_t)


print("--- TTA'siz (single-scale) ---")
p1_no, t1_no = run_eval_tta(test1_loader, tta=False)
p2_no, t2_no = run_eval_tta(test2_loader, tta=False)
m1_no = compute_metrics(p1_no, t1_no)
m2_no = compute_metrics(p2_no, t2_no)
print(f"  Test1 mIoU: {m1_no['mIoU']*100:.2f}%  |  Test2 mIoU: {m2_no['mIoU']*100:.2f}%")

print("\n--- Multi-scale TTA (HFlip + polarity + scale 0.75/1.0/1.25) ---")
p1, t1 = run_eval_tta(test1_loader, tta=True)
p2, t2 = run_eval_tta(test2_loader, tta=True)
metrics1    = compute_metrics(p1, t1)
metrics2    = compute_metrics(p2, t2)
metrics_all = compute_metrics(np.concatenate([p1,p2]), np.concatenate([t1,t2]))

print_metrics(metrics1,    "Test1 (Inline) — Multi-scale TTA")
print_metrics(metrics2,    "Test2 (Crossline — Genelleme) — Multi-scale TTA")
print_metrics(metrics_all, "Birlesik Test1 + Test2 — Multi-scale TTA")

results = {
    "model": "DeepLabV3+ v9 (5-channel 2.5D + Lovasz + xline-aware aug + 384x384 + multi-scale TTA)",
    "split": {
        "val_inline_range": [int(val_start), int(val_end)],
        "buffer": int(BUFFER),
        "n_train_inline": int(len(train_inline_idx)),
        "n_val_inline": int(len(val_inline_idx)),
        "n_train_xline": int(len(train_xline_idx)),
    },
    "img_size": list(IMG_SIZE),
    "tta_scales": [1.0] + TTA_SCALES,
    "test1": metrics1, "test2": metrics2, "combined": metrics_all,
    "test1_no_tta": m1_no, "test2_no_tta": m2_no,
    "history": history,
    "config": {
        "n_neighbors": int(N_NEIGHBORS),
        "n_channels": int(N_CHANNELS),
        "loss_weights": {"ls_ce": 0.30, "dice": 0.25, "focal": 0.25, "lovasz": 0.20},
        "xline_aware_aug": True,
        "multi_scale_tta": True,
    },
}
out_path = METRICS_DIR / "deeplabv3plus_v9_metrics.json"
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nMetrikler kaydedildi: {out_path}")


En iyi model yuklendi: C:\Users\Teknogenetik\Desktop\sismik-proje-main\checkpoints_v7\deeplabv3plus_v9_best.pth
--- TTA'siz (single-scale) ---
  Test1 mIoU: 47.53%  |  Test2 mIoU: 47.10%

--- Multi-scale TTA (HFlip + polarity + scale 0.75/1.0/1.25) ---

 Test1 (Inline) — Multi-scale TTA
  PA        : 75.10%
  MCA       : 66.60%
  mIoU      : 48.36%
  Mean Dice : 63.19%
  Per-class IoU:
    S0 Upper NS            : IoU=0.5623  Dice=0.7199
    S1 Lower NS            : IoU=0.5939  Dice=0.7452
    S2 Rijnland            : IoU=0.7969  Dice=0.8869
    S3 Scruff              : IoU=0.2836  Dice=0.4419
    S4 Zechstein           : IoU=0.3185  Dice=0.4832
    S5 Under Zech          : IoU=0.3463  Dice=0.5144

 Test2 (Crossline — Genelleme) — Multi-scale TTA
  PA        : 84.91%
  MCA       : 62.22%
  mIoU      : 47.80%
  Mean Dice : 58.97%
  Per-class IoU:
    S0 Upper NS            : IoU=0.7317  Dice=0.8451
    S1 Lower NS            : IoU=0.5960  Dice=0.7468
    S2 Rijnland            : IoU=0.8

## 10. Egitim Egrileri

In [ ]:
epochs_x = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(epochs_x, history['train_loss'], label='Train', color='steelblue')
axes[0].plot(epochs_x, history['val_loss'],   label='Val',   color='coral')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_x, history['val_miou'], label='mIoU',  color='green')
axes[1].plot(epochs_x, history['val_dice'], label='Dice',  color='purple')
axes[1].set_title('Metrikler'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(epochs_x, history['lr'], color='darkorange')
axes[2].set_title('Learning Rate'); axes[2].grid(alpha=0.3)

for ax in axes: ax.set_xlabel('Epoch')
plt.suptitle("DeepLabV3+ v9 — Egitim Sureci (5-ch 2.5D + 384x384 + Lovasz + multi-scale TTA)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "training_curves_v9.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Segmentasyon Karsilastirmasi (Test1 + Test2)

In [ ]:
model.eval()

fig, axes = plt.subplots(6, 3, figsize=(15, 24))

test1_sample = np.linspace(0, len(test1_ds)-1, 3, dtype=int)
test2_sample = np.linspace(0, len(test2_ds)-1, 3, dtype=int)

with torch.no_grad():
    for row, (ds, idx, label) in enumerate(
        [(test1_ds, i, f"Inline #{i}") for i in test1_sample] +
        [(test2_ds, i, f"Crossline #{i}") for i in test2_sample]
    ):
        img, mask = ds[idx]
        inp = img.unsqueeze(0).to(device)
        if use_amp:
            with torch.amp.autocast("cuda"):
                pred = model(inp).argmax(1).squeeze().cpu().numpy()
        else:
            pred = model(inp).argmax(1).squeeze().cpu().numpy()

        axes[row, 0].imshow(img[1].numpy(), cmap="seismic", vmin=-2, vmax=2, aspect="auto")
        axes[row, 0].set_title(f"Sismik — {label}", fontsize=9); axes[row, 0].axis("off")

        axes[row, 1].imshow(mask.numpy(), cmap=cmap_facies, vmin=-0.5, vmax=5.5, aspect="auto")
        axes[row, 1].set_title("Gercek Fasiyes", fontsize=9); axes[row, 1].axis("off")

        axes[row, 2].imshow(pred, cmap=cmap_facies, vmin=-0.5, vmax=5.5, aspect="auto")
        axes[row, 2].set_title("v9 Tahmini", fontsize=9); axes[row, 2].axis("off")

patches_vis = [mpatches.Patch(color=FACIES_COLORS[j], label=f"S{j}: {CLASS_NAMES[j]}") for j in range(NUM_CLASSES)]
fig.legend(handles=patches_vis, loc="lower center", ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.01))
plt.suptitle("DeepLabV3+ v9 — Inline + Crossline Segmentasyon", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "segmentation_comparison_v9.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Confusion Matrix

In [ ]:
cm_arr  = np.array(metrics_all['confusion_matrix'])
cm_norm = cm_arr.astype(float) / cm_arr.sum(axis=1, keepdims=True).clip(min=1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, data, title in zip(axes, [cm_arr, cm_norm], ["Ham Sayilar", "Normalize (satir %)"]):
    im = ax.imshow(data, cmap="Blues")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels([f"S{j}\n{CLASS_NAMES[j]}" for j in range(NUM_CLASSES)], fontsize=8, rotation=15)
    ax.set_yticklabels([f"S{j} {CLASS_NAMES[j]}" for j in range(NUM_CLASSES)], fontsize=8)
    ax.set_xlabel("Tahmin"); ax.set_ylabel("Gercek")
    ax.set_title(f"Confusion Matrix — {title}", fontsize=11)
    plt.colorbar(im, ax=ax)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            val = f"{data[i,j]:.2f}" if title != "Ham Sayilar" else f"{int(data[i,j]):,}"
            ax.text(j, i, val, ha="center", va="center", fontsize=6,
                    color="white" if data[i,j] > data.max()*0.5 else "black")

plt.suptitle("DeepLabV3+ v9 — Confusion Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "confusion_matrix_v9.png", dpi=150, bbox_inches="tight")
plt.show()

## 13. Per-class IoU / Dice

In [ ]:
iou_vals  = metrics_all['per_class_iou']
dice_vals = metrics_all['per_class_dice']
x = np.arange(NUM_CLASSES); width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - width/2, iou_vals,  width, label="IoU",  color=FACIES_COLORS, alpha=0.85, edgecolor="k", lw=0.5)
b2 = ax.bar(x + width/2, dice_vals, width, label="Dice", color=FACIES_COLORS, alpha=0.55, edgecolor="k", lw=0.5, hatch="///")

for bar, v in zip(b1, iou_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f"{v:.3f}", ha="center", fontsize=8)
for bar, v in zip(b2, dice_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f"{v:.3f}", ha="center", fontsize=8)

miou_val  = metrics_all['mIoU']
mdice_val = metrics_all['mean_dice']
ax.axhline(miou_val,  color="green",  ls="--", lw=1.2, label=f"mIoU={miou_val:.3f}")
ax.axhline(mdice_val, color="purple", ls="--", lw=1.2, label=f"mDice={mdice_val:.3f}")

ax.set_xticks(x)
ax.set_xticklabels([f"S{j}\n{CLASS_NAMES[j]}" for j in range(NUM_CLASSES)], fontsize=9)
ax.set_ylabel("Skor"); ax.set_ylim(0, 1.1)
ax.set_title("Per-class IoU ve Dice (Birlesik Test) — v9", fontsize=11)
ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "per_class_metrics_v9.png", dpi=150, bbox_inches="tight")
plt.show()

## 14. Versiyon Karsilastirma (v3 / v5 / v7-broken / v7-fixed)
Methodology fix oncesi ve sonrasi sayilarini yan yana goster — sunum icin "oncesi/sonrasi" tablosu.

In [ ]:
v3_metrics_path        = PROJECT_DIR / "results" / "metrics" / "deeplabv3plus_v3_metrics.json"
v5_metrics_path        = PROJECT_DIR / "results" / "metrics" / "deeplabv3plus_v5_metrics.json"
v7_broken_metrics_path = PROJECT_DIR / "results" / "metrics" / "deeplabv3plus_v7_metrics.json"
v7_fixed_metrics_path  = PROJECT_DIR / "results" / "metrics" / "deeplabv3plus_v7_fixed_metrics.json"
v7_c4fix_metrics_path  = PROJECT_DIR / "results" / "metrics" / "deeplabv3plus_v7_c4fix_metrics.json"

print("=" * 100)
print(" VERSIYON EVRIMI — v3 (broken) -> v5 -> v7-broken -> v7-fixed -> v7-c4fix -> v9")
print("=" * 100)

def metrics_summary(d):
    return {
        "test1_miou":    d['test1']['mIoU'],
        "test2_miou":    d['test2']['mIoU'],
        "combined_miou": d['combined']['mIoU'],
        "combined_dice": d['combined']['mean_dice'],
        "combined_pa":   d['combined']['PA'],
        "combined_mca":  d['combined']['MCA'],
    }

# v9 (bu notebook'tan)
v9 = metrics_summary({
    'test1': metrics1, 'test2': metrics2, 'combined': metrics_all,
})

# Diger surumleri JSON'dan yukle (yoksa cached defaults)
def _load(path, defaults, label):
    if path.exists():
        with open(path) as f:
            d = json.load(f)
        print(f"  {label} JSON yuklendi.")
        return metrics_summary(d)
    print(f"  {label} cached degerler.")
    return defaults

v7_c4fix  = _load(v7_c4fix_metrics_path,
                  {"test1_miou": 0.7902, "test2_miou": 0.6668, "combined_miou": 0.7779,
                   "combined_dice": 0.8672, "combined_pa": 0.9369, "combined_mca": 0.8615}, "v7-c4fix")
v7_fixed  = _load(v7_fixed_metrics_path,
                  {"test1_miou": 0.7625, "test2_miou": 0.6681, "combined_miou": 0.7668,
                   "combined_dice": 0.8603, "combined_pa": 0.9319, "combined_mca": 0.8615}, "v7-fixed")
v7_broken = _load(v7_broken_metrics_path,
                  {"test1_miou": 0.7881, "test2_miou": 0.6986, "combined_miou": 0.7926,
                   "combined_dice": 0.8775, "combined_pa": 0.9413, "combined_mca": 0.8975}, "v7-broken")
v5        = _load(v5_metrics_path,
                  {"test1_miou": 0.7577, "test2_miou": 0.6585, "combined_miou": 0.7582,
                   "combined_dice": 0.8536, "combined_pa": 0.9286, "combined_mca": 0.8756}, "v5")
v3        = _load(v3_metrics_path,
                  {"test1_miou": 0.6446, "test2_miou": 0.2710, "combined_miou": 0.4012,
                   "combined_dice": 0.5453, "combined_pa": 0.6953, "combined_mca": 0.0}, "v3")

print()
fmt = "{:30s} | {:>8s} | {:>8s} | {:>10s} | {:>10s} | {:>10s} | {:>8s}"
print(fmt.format("Metrik", "v3", "v5", "v7-broken", "v7-fixed", "v7-c4fix", "v9"))
print("-" * 100)
for key, label in [
    ("test1_miou",    "Test1 mIoU (inline)"),
    ("test2_miou",    "Test2 mIoU (crossline)"),
    ("combined_miou", "Combined mIoU"),
    ("combined_dice", "Combined Dice"),
    ("combined_pa",   "Combined PA"),
]:
    row = [v3[key], v5[key], v7_broken[key], v7_fixed[key], v7_c4fix[key], v9[key]]
    best = max(row)
    marker = " <- v9 best" if v9[key] >= best else ""
    print(fmt.format(label,
                     f"{row[0]:.4f}", f"{row[1]:.4f}",
                     f"{row[2]:.4f}", f"{row[3]:.4f}",
                     f"{row[4]:.4f}", f"{row[5]:.4f}") + marker)
print("=" * 100)

# v9'un methodology-fix'li 3 surume gore deltasi
print(f"\nv9 vs v7-fixed (methodology-fix baseline):")
for key, label in [
    ("test1_miou",    "Test1 mIoU"),
    ("test2_miou",    "Test2 mIoU"),
    ("combined_miou", "Combined mIoU"),
    ("combined_dice", "Combined Dice"),
]:
    delta = v9[key] - v7_fixed[key]
    sign = "+" if delta >= 0 else ""
    print(f"  {label:25s}: {sign}{delta*100:+.2f} puan")

print(f"\nv9 vs v7-c4fix (Class 4 paket baseline):")
for key, label in [
    ("test1_miou",    "Test1 mIoU"),
    ("test2_miou",    "Test2 mIoU"),
    ("combined_miou", "Combined mIoU"),
    ("combined_dice", "Combined Dice"),
]:
    delta = v9[key] - v7_c4fix[key]
    sign = "+" if delta >= 0 else ""
    print(f"  {label:25s}: {sign}{delta*100:+.2f} puan")

print(f"\nPer-class IoU (Combined) — v9:")
for i, name in enumerate(CLASS_NAMES):
    iou = metrics_all['per_class_iou'][i]
    dice = metrics_all['per_class_dice'][i]
    print(f"  S{i} {name:20s}: IoU={iou:.4f}  Dice={dice:.4f}")

# Class 4 hedef takibi
class4_t2 = metrics2["per_class_iou"][4]
class4_delta_fixed = (class4_t2 - 0.1177) * 100
print(f"\nClass 4 (Zechstein) Test2 IoU evrimi (sunum highlight):")
print(f"  v7-fixed   : 0.1177  (baseline)")
print(f"  v7-c4fix   : 0.1820  (+6.43p, goreli +%55)")
print(f"  v9         : {class4_t2:.4f}  ({class4_delta_fixed:+.2f}p vs v7-fixed)")

print(f"\n{'=' * 100}")
print(" v9 FINAL SONUCLAR")
print(f"{'=' * 100}")
print(f"  Test1 mIoU     : {metrics1['mIoU']*100:.2f}%  (inline)")
print(f"  Test2 mIoU     : {metrics2['mIoU']*100:.2f}%  (crossline — genelleme)")
print(f"  Combined mIoU  : {metrics_all['mIoU']*100:.2f}%")
print(f"  Combined Dice  : {metrics_all['mean_dice']*100:.2f}%")
print(f"  Combined PA    : {metrics_all['PA']*100:.2f}%")
print(f"  Combined MCA   : {metrics_all['MCA']*100:.2f}%")
print(f"  Class 4 Test2  : {metrics2['per_class_iou'][4]*100:.2f}%  (key generalization metric)")
print(f"\n  v9 = EfficientNet-B4 + 5-channel 2.5D + 384x384 + Lovasz-Softmax + xline-aware aug + multi-scale TTA")
print(f"        on Yol A methodology-fix split")
print(f"\n  Model : {BEST_MODEL_PATH}")
print(f"  JSON  : {METRICS_DIR}/deeplabv3plus_v9_metrics.json")
